# Four coffee-shop drinks: sales index through the year

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Vertical Line Graph - Changing the Data Range** · a worked example from the StatNex teaching guide

**The story.** A coffee chain tracks a monthly sales index for four drinks (hot coffee, iced coffee, hot chocolate and smoothie), where 100 is a normal month for that drink. The table also has two dummy columns, helper columns from a spreadsheet version of the chart that are not data. This example leaves them out and charts only the four drinks.

**The question.** How differently do the four drinks move through the year?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `11_chart/vertical_line_graph_range.csv` (12 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Month,Hot coffee,Iced coffee,Hot chocolate,Smoothie,Dummy 1,Dummy 2
January,128.0,46.0,168.0,70.0,0.5,0.0
February,124.0,50.0,150.0,74.0,1.5,0.0
March,112.0,66.0,112.0,88.0,2.5,0.0
April,98.0,92.0,74.0,102.0,3.5,0.0
May,88.0,118.0,50.0,116.0,4.5,0.0
June,80.0,142.0,38.0,128.0,5.5,0.0
July,74.0,160.0,32.0,136.0,6.5,0.0
August,76.0,158.0,34.0,132.0,7.5,0.0
September,90.0,124.0,52.0,112.0,8.5,0.0
October,106.0,82.0,96.0,92.0,9.5,0.0
November,120.0,60.0,136.0,78.0,10.5,0.0
December,134.0,48.0,196.0,72.0,11.5,0.0

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per item, one column per series. Text and blank cells are missing; every
# item needs at least one number.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
series = ["Hot coffee", "Iced coffee", "Hot chocolate", "Smoothie"]
item_name = "Month"
items = data[item_name].astype(str)
values = data[series].apply(pd.to_numeric, errors="coerce")

## Data for the chart

In [ ]:
# Dummy 1 and Dummy 2 are the scaffolding Excel needs to draw this chart: each item's
# position counted from the top (0.5, 1.5, ...) and the baseline of the bars.
table = values.copy()
table.insert(0, item_name, items)
table["Dummy 1"] = np.arange(len(table)) + 0.5
table["Dummy 2"] = 0
print(table.to_string(index=False))

## Spread between the series

In [ ]:
# For each item, the lowest and highest series and the gap between them -- the width
# between the lines on the chart -- largest gap first. Blank cells are skipped. When
# the series are group averages, a gap shows where the averages differ, not whether
# the difference is larger than chance: that needs the spread within each group and
# a test.
spread = pd.DataFrame({"item": items, "minimum": values.min(axis=1),
                       "maximum": values.max(axis=1)})
spread["spread"] = spread["maximum"] - spread["minimum"]
spread["highest series"] = values.idxmax(axis=1)
spread = spread.iloc[np.argsort(spread["spread"].to_numpy())[::-1]]
print()
print(spread.to_string(index=False))

## Vertical line graph

In [ ]:
# Items run down the page in the order of the table, values across.
Path("figures").mkdir(exist_ok=True)
positions = np.arange(len(items))
fig, ax = plt.subplots(figsize=(7, 0.45 * len(items) + 1.5))
for name in series:
    ax.plot(values[name], positions, marker="o", label=name)
ax.set_yticks(positions, items)
ax.invert_yaxis()
ax.legend(fontsize=8)
fig.savefig("figures/vertical_line_graph.png", dpi=120, bbox_inches="tight")

## What the result says

Each line is one drink, and each row is one month. An index of 100 is a normal month for that drink. Hot chocolate goes from 196 in December down to 32 in July. Iced coffee does the opposite: 160 in July and 46 in January. Hot coffee stays between 74 and 134, and smoothies between 70 and 136.

The spread is the gap between the highest and the lowest drink in one month. It is largest in December (148) and smallest in October (24) and April (28), when the lines cross.

Hot chocolate and iced coffee are strongly seasonal and mirror each other. Hot coffee and smoothies swing far less. The chart compares each drink's ups and downs, not how many cups each one sells.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.